# Homework 2 — Analisis Order E-commerce + Kalkulator Revenue

**Versi MENTOR (lengkap).**

Latihan mandiri setelah Sesi 2. Menggabungkan **Pandas** (analisis data) dengan
**Python Fundamentals** (function, dict manipulation) — dua skill yang sering dipakai
bersama di dunia nyata.

## Skenario
Anda pegang data 200 order dari toko e-commerce (`orders_hw.csv`).
Tugas Anda: jawab pertanyaan bisnis + bangun beberapa function kalkulator yang bisa
dipakai ulang.

## Kolom di `orders_hw.csv`
- `order_id` — ID unik order (string)
- `customer` — nama customer
- `product` — nama produk
- `category` — kategori: Elektronik / Fashion / Makanan / Kesehatan
- `price` — harga satuan (IDR)
- `quantity` — jumlah dibeli
- `order_date` — tanggal order (ISO)
- `city` — kota pengiriman

Perkiraan waktu: 30–45 menit.


## Setup


In [4]:
import pandas as pd

df = pd.read_csv("orders_hw_csv.csv")
print("Shape:", df.shape)
df.head()


Shape: (200, 8)


,order_id,customer,product,category,price,quantity,order_date,city
0,ORD-0001,Citra Pratama,Suplemen,Kesehatan,176611,5,2024-07-22,Makassar
1,ORD-0002,Indah Utami,Snack,Makanan,93381,8,2024-08-30,Makassar
2,ORD-0003,Eko Santoso,Sepatu,Fashion,347289,2,2024-12-05,Medan
3,ORD-0004,Lestari Permana,Suplemen,Kesehatan,63894,4,2024-09-14,Jakarta
4,ORD-0005,Kartika Lestari,Biskuit,Makanan,71064,2,2024-12-03,Surabaya


---
## Task 1 — Inspeksi Awal
Sebelum analisis apa pun, kenali dulu datanya.


In [ ]:
# Info tipe kolom + missing values
df.info()


In [ ]:
# Statistik ringkas kolom numerik
df.describe()


---
## Task 2 — Berapa Order per Kota?
Pakai `.value_counts()` pada kolom `city`.


In [ ]:
df["city"].value_counts()


---
## Task 3 — Total Revenue per Kategori
Revenue = price × quantity. Jumlahkan per kategori, urut dari tertinggi.

**Catatan:** untuk sementara pakai perhitungan sederhana `price * quantity`.
Di Task 4 kita bikin function kalkulator yang lebih fleksibel (bisa handle diskon).


In [ ]:
df["revenue_simple"] = df["price"] * df["quantity"]

revenue_per_kategori = df.groupby("category")["revenue_simple"].sum().sort_values(ascending=False)
print(revenue_per_kategori)


---
## Task 4 — Bangun Function Kalkulator: `hitung_revenue`
Function yang menghitung revenue sebuah order, dengan diskon opsional (default 0).

Signature: `hitung_revenue(price, qty, diskon=0) -> float`

Rumus: `revenue = price * qty * (1 - diskon)`


In [ ]:
def hitung_revenue(price: float, qty: int, diskon: float = 0.0) -> float:
    """Hitung revenue satu order dengan diskon opsional."""
    return price * qty * (1 - diskon)

# Test
print(hitung_revenue(25000, 3))            # 75000 (tanpa diskon)
print(hitung_revenue(25000, 3, 0.1))       # 67500 (diskon 10%)


---
## Task 5 — Terapkan Function ke DataFrame
Buat kolom baru `revenue` di df, dihitung dengan `hitung_revenue()`.
Gunakan diskon 0 untuk sekarang (bisa diubah nanti).

Ada dua cara umum:
- **Vectorized** (cepat, disarankan): `df["revenue"] = hitung_revenue(df["price"], df["quantity"])`
- **apply per baris** (lebih fleksibel): `df.apply(lambda r: hitung_revenue(r["price"], r["quantity"]), axis=1)`


In [ ]:
# Cara vectorized — paling cepat, cocok untuk function sederhana
df["revenue"] = hitung_revenue(df["price"], df["quantity"])

df[["order_id", "customer", "price", "quantity", "revenue"]].head()


---
## Task 6 — Function `count_orders_per_customer`
Function yang menerima DataFrame, mengembalikan **dict** `{customer: jumlah_order}`.

Ini contoh fusi Pandas + Python — outputnya dict, bukan Series, sehingga bisa diakses seperti dict biasa.


In [ ]:
def count_orders_per_customer(df: pd.DataFrame) -> dict:
    """Hitung jumlah order per customer."""
    counts = df["customer"].value_counts()
    return counts.to_dict()

order_counts = count_orders_per_customer(df)
print("Type:", type(order_counts))

# Contoh akses seperti dict biasa
for nama, jumlah in list(order_counts.items())[:5]:
    print(f"  {nama}: {jumlah} order")


---
## Task 7 — Siapa Customer dengan Total Belanja Terbesar?
Gunakan kolom `revenue` yang sudah dibuat di Task 5.
Group by customer, jumlahkan revenue, ambil yang paling besar.


In [ ]:
spending_per_customer = df.groupby("customer")["revenue"].sum().sort_values(ascending=False)
print(spending_per_customer.head())
print()
print(f"Top spender: {spending_per_customer.idxmax()}")
print(f"Total belanja: Rp {spending_per_customer.max():,.0f}")


---
## Bonus — Simpan Hasil Analisis ke CSV
Tambahkan kolom-kolom hasil analisis dan simpan.


In [ ]:
# Pilih kolom yang mau disimpan
hasil = df[["order_id", "customer", "category", "price", "quantity", "revenue", "city"]]
hasil.to_csv("hasil_analisis.csv", index=False)
print("Tersimpan sebagai hasil_analisis.csv")


---
## Cek Diri Sendiri
Setelah selesai, pastikan:
- `df["revenue"]` sudah ada dan nilainya benar (spot-check: `df.head()`)
- Function `hitung_revenue` bekerja dengan dan tanpa diskon
- `count_orders_per_customer(df)` menghasilkan dict, bukan Series
- Top spender bisa dijawab dengan satu perintah `spending_per_customer.idxmax()`
- File `hasil_analisis.csv` ada di folder yang sama

Bagus! Ini adalah pola kerja yang persis kita pakai untuk pipeline data di Sesi 3.
